In [1]:
# ── CELL 1: Import AIF360 ─────────────────────────
import pandas as pd
import numpy as np
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

print("AIF360 imported successfully")

pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


AIF360 imported successfully


In [3]:
# ── CELL 2: Load Adult Dataset for AIF360 ─────────
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['age','workclass','fnlwgt','education','education-num',
           'marital-status','occupation','relationship','race',
           'sex','capital-gain','capital-loss','hours-per-week',
           'native-country','income']
df = pd.read_csv(url, names=columns, na_values=' ?',
                  skipinitialspace=True).dropna()

# Encode
df['income_binary'] = (df['income']=='>50K').astype(int)
df['sex_binary'] = (df['sex']=='Male').astype(int)
cat_cols = ['workclass','education','marital-status','occupation',
            'relationship','race','native-country']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
df = df.drop(['income','sex','fnlwgt'], axis=1)

# Create AIF360 BinaryLabelDataset
aif_adult = BinaryLabelDataset(
    df=df,
    label_names=['income_binary'],
    protected_attribute_names=['sex_binary'],
    favorable_label=1,
    unfavorable_label=0
)

print("AIF360 Adult dataset created successfully")
print("Privileged group:   Male (sex_binary=1)")
print("Unprivileged group: Female (sex_binary=0)")

AIF360 Adult dataset created successfully
Privileged group:   Male (sex_binary=1)
Unprivileged group: Female (sex_binary=0)


In [4]:
# ── CELL 3: Dataset Fairness Metrics ──────────────
privileged   = [{'sex_binary': 1}]  # Male
unprivileged = [{'sex_binary': 0}]  # Female

metric_adult = BinaryLabelDatasetMetric(
    aif_adult,
    privileged_groups=privileged,
    unprivileged_groups=unprivileged
)

spd = metric_adult.statistical_parity_difference()
di  = metric_adult.disparate_impact()

print("=== ADULT DATASET FAIRNESS METRICS ===")
print(f"Statistical Parity Difference: {spd:.4f}")
print(f"Disparate Impact Ratio:        {di:.4f}")
print()
if abs(spd) > 0.1:
    print("⚠️  SPD > 0.1 — SIGNIFICANT BIAS DETECTED")
if di < 0.8:
    print("⚠️  DI < 0.8  — SIGNIFICANT BIAS DETECTED")

=== ADULT DATASET FAIRNESS METRICS ===
Statistical Parity Difference: -0.1963
Disparate Impact Ratio:        0.3580

⚠️  SPD > 0.1 — SIGNIFICANT BIAS DETECTED
⚠️  DI < 0.8  — SIGNIFICANT BIAS DETECTED


In [5]:
# ── CELL 4: Train Model and Measure Classification Fairness
adult_train, adult_test = aif_adult.split([0.8], shuffle=True, seed=42)

X_train = adult_train.features
y_train = adult_train.labels.ravel()
X_test  = adult_test.features
y_test  = adult_test.labels.ravel()

model = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

adult_pred = adult_test.copy()
adult_pred.labels = y_pred.reshape(-1,1)

class_metric = ClassificationMetric(
    adult_test, adult_pred,
    privileged_groups=privileged,
    unprivileged_groups=unprivileged
)

eod = class_metric.equal_opportunity_difference()
aod = class_metric.average_odds_difference()
acc = class_metric.accuracy()

print("=== CLASSIFICATION FAIRNESS METRICS — Adult (XGBoost) ===")
print(f"Accuracy:                      {acc*100:.2f}%")
print(f"Equal Opportunity Difference:  {eod:.4f}")
print(f"Average Odds Difference:       {aod:.4f}")
print(f"Statistical Parity Difference: {spd:.4f}")
print(f"Disparate Impact Ratio:        {di:.4f}")
print()
if abs(eod) > 0.1:
    print(f"⚠️  EOD = {eod:.4f} — BIAS DETECTED in Equal Opportunity")
if abs(aod) > 0.1:
    print(f"⚠️  AOD = {aod:.4f} — BIAS DETECTED in Average Odds")
print("\n✅ Fairness metrics computed successfully")

=== CLASSIFICATION FAIRNESS METRICS — Adult (XGBoost) ===
Accuracy:                      87.10%
Equal Opportunity Difference:  -0.0957
Average Odds Difference:       -0.0835
Statistical Parity Difference: -0.1963
Disparate Impact Ratio:        0.3580


✅ Fairness metrics computed successfully


In [6]:
# ── CELL 5: Fairness Results Summary Table ─────────
fairness_results = pd.DataFrame([
    {
        'Dataset': 'Adult',
        'Model': 'XGBoost',
        'Protected': 'Gender',
        'Privileged': 'Male',
        'Unprivileged': 'Female',
        'SPD': round(spd, 4),
        'DI': round(di, 4),
        'EOD': round(eod, 4),
        'AOD': round(aod, 4),
        'Bias Detected': 'YES'
    }
])

print("=== FAIRNESS METRICS SUMMARY TABLE ===")
print(fairness_results.to_string(index=False))

fairness_results.to_csv(
    r'C:\Users\niyat\Desktop\pr\ML-Bias-Fairness\results\fairness_metrics.csv',
    index=False)
print("\nFairness results saved to CSV")
print("\n✅ ADULT DATASET FAIRNESS ANALYSIS COMPLETE")
print("\nKey findings:")
print(f"  SPD = {spd:.4f} → Females 19.6% less likely to be predicted >$50K")
print(f"  DI  = {di:.4f}  → Females receive positive outcome at only 35.8% rate of males")
print(f"  EOD = {eod:.4f} → Unequal true positive rates between gender groups")
print(f"  AOD = {aod:.4f} → Unequal overall odds between gender groups")

=== FAIRNESS METRICS SUMMARY TABLE ===
Dataset   Model Protected Privileged Unprivileged     SPD    DI     EOD     AOD Bias Detected
  Adult XGBoost    Gender       Male       Female -0.1963 0.358 -0.0957 -0.0835           YES

Fairness results saved to CSV

✅ ADULT DATASET FAIRNESS ANALYSIS COMPLETE

Key findings:
  SPD = -0.1963 → Females 19.6% less likely to be predicted >$50K
  DI  = 0.3580  → Females receive positive outcome at only 35.8% rate of males
  EOD = -0.0957 → Unequal true positive rates between gender groups
  AOD = -0.0835 → Unequal overall odds between gender groups


In [7]:
# ── CELL 6: German Credit Fairness Metrics ─────────
url_g = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
col_names = ['status','duration','credit_history','purpose','credit_amount',
             'savings','employment','installment_rate','personal_status_sex',
             'other_debtors','residence','property','age','other_installments',
             'housing','existing_credits','job','dependents','telephone',
             'foreign_worker','target']
df_g = pd.read_csv(url_g, sep=' ', header=None, names=col_names)

gender_map = {'A91':'Male','A92':'Female','A93':'Male',
              'A94':'Male','A95':'Female'}
df_g['gender_binary'] = (df_g['personal_status_sex'].map(gender_map)=='Male').astype(int)
df_g['target_binary'] = (df_g['target']==2).astype(int)

cat_cols_g = ['status','credit_history','purpose','savings','employment',
              'personal_status_sex','other_debtors','property',
              'other_installments','housing','job','telephone','foreign_worker']
le = LabelEncoder()
for col in cat_cols_g:
    df_g[col] = le.fit_transform(df_g[col].astype(str))
df_g = df_g.drop('target', axis=1)

# Create AIF360 dataset
aif_german = BinaryLabelDataset(
    df=df_g,
    label_names=['target_binary'],
    protected_attribute_names=['gender_binary'],
    favorable_label=0,
    unfavorable_label=1
)

priv_g   = [{'gender_binary': 1}]
unpriv_g = [{'gender_binary': 0}]

metric_g = BinaryLabelDatasetMetric(
    aif_german,
    privileged_groups=priv_g,
    unprivileged_groups=unpriv_g
)

spd_g = metric_g.statistical_parity_difference()
di_g  = metric_g.disparate_impact()

# Train and predict
g_train, g_test = aif_german.split([0.8], shuffle=True, seed=42)
model_g = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)
model_g.fit(g_train.features, g_train.labels.ravel())
y_pred_g = model_g.predict(g_test.features)
g_pred = g_test.copy()
g_pred.labels = y_pred_g.reshape(-1,1)

class_g = ClassificationMetric(g_test, g_pred,
    privileged_groups=priv_g,
    unprivileged_groups=unpriv_g)

eod_g = class_g.equal_opportunity_difference()
aod_g = class_g.average_odds_difference()
acc_g = class_g.accuracy()

print("=== GERMAN CREDIT FAIRNESS METRICS (XGBoost) ===")
print(f"Accuracy:                      {acc_g*100:.2f}%")
print(f"Statistical Parity Difference: {spd_g:.4f}")
print(f"Disparate Impact Ratio:        {di_g:.4f}")
print(f"Equal Opportunity Difference:  {eod_g:.4f}")
print(f"Average Odds Difference:       {aod_g:.4f}")
print()
for val, name in [(spd_g,'SPD'),(eod_g,'EOD'),(aod_g,'AOD')]:
    if abs(val) > 0.1:
        print(f"⚠️  {name} = {val:.4f} — BIAS DETECTED")
if di_g < 0.8:
    print(f"⚠️  DI = {di_g:.4f} — BIAS DETECTED")
print("\n✅ German Credit fairness metrics complete")

=== GERMAN CREDIT FAIRNESS METRICS (XGBoost) ===
Accuracy:                      75.00%
Statistical Parity Difference: -0.0748
Disparate Impact Ratio:        0.8966
Equal Opportunity Difference:  -0.0667
Average Odds Difference:       -0.1444

⚠️  AOD = -0.1444 — BIAS DETECTED

✅ German Credit fairness metrics complete


In [8]:
# ── CELL 7: Law School Fairness Metrics ────────────
df_l = pd.read_csv(
    r'C:\Users\niyat\Desktop\pr\ML-Bias-Fairness\data\law_school\law_dataset.csv')
df_l = df_l.dropna(subset=['pass_bar','male','racetxt','lsat','ugpa'])
le_race = LabelEncoder()
df_l['race_encoded'] = le_race.fit_transform(df_l['racetxt'].astype(str))
features = ['lsat','ugpa','zfygpa','zgpa','fulltime',
            'fam_inc','male','race_encoded','tier']
df_l_clean = df_l[features+['pass_bar']].dropna()

aif_law = BinaryLabelDataset(
    df=df_l_clean,
    label_names=['pass_bar'],
    protected_attribute_names=['male'],
    favorable_label=1,
    unfavorable_label=0
)

priv_l   = [{'male': 1}]
unpriv_l = [{'male': 0}]

metric_l = BinaryLabelDatasetMetric(
    aif_law,
    privileged_groups=priv_l,
    unprivileged_groups=unpriv_l
)

spd_l = metric_l.statistical_parity_difference()
di_l  = metric_l.disparate_impact()

l_train, l_test = aif_law.split([0.8], shuffle=True, seed=42)
model_l = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)
model_l.fit(l_train.features, l_train.labels.ravel())
y_pred_l = model_l.predict(l_test.features)
l_pred = l_test.copy()
l_pred.labels = y_pred_l.reshape(-1,1)

class_l = ClassificationMetric(l_test, l_pred,
    privileged_groups=priv_l,
    unprivileged_groups=unpriv_l)

eod_l = class_l.equal_opportunity_difference()
aod_l = class_l.average_odds_difference()
acc_l = class_l.accuracy()

print("=== LAW SCHOOL FAIRNESS METRICS (XGBoost) ===")
print(f"Accuracy:                      {acc_l*100:.2f}%")
print(f"Statistical Parity Difference: {spd_l:.4f}")
print(f"Disparate Impact Ratio:        {di_l:.4f}")
print(f"Equal Opportunity Difference:  {eod_l:.4f}")
print(f"Average Odds Difference:       {aod_l:.4f}")
print()
for val, name in [(spd_l,'SPD'),(eod_l,'EOD'),(aod_l,'AOD')]:
    if abs(val) > 0.1:
        print(f"⚠️  {name} = {val:.4f} — BIAS DETECTED")
if di_l < 0.8:
    print(f"⚠️  DI = {di_l:.4f} — BIAS DETECTED")
print("\n✅ Law School fairness metrics complete")

=== LAW SCHOOL FAIRNESS METRICS (XGBoost) ===
Accuracy:                      90.64%
Statistical Parity Difference: -0.0212
Disparate Impact Ratio:        0.9768
Equal Opportunity Difference:  -0.0006
Average Odds Difference:       -0.0203


✅ Law School fairness metrics complete


In [9]:
# ── CELL 8: Complete Fairness Summary ──────────────
all_fairness = pd.DataFrame([
    {'Dataset':'Adult',         'Model':'XGBoost', 'Protected':'Gender',
     'SPD':round(spd,4),   'DI':round(di,4),
     'EOD':round(eod,4),   'AOD':round(aod,4)},
    {'Dataset':'German Credit', 'Model':'XGBoost', 'Protected':'Gender',
     'SPD':round(spd_g,4), 'DI':round(di_g,4),
     'EOD':round(eod_g,4), 'AOD':round(aod_g,4)},
    {'Dataset':'Law School',    'Model':'XGBoost', 'Protected':'Gender',
     'SPD':round(spd_l,4), 'DI':round(di_l,4),
     'EOD':round(eod_l,4), 'AOD':round(aod_l,4)},
])

print("=== COMPLETE FAIRNESS METRICS ACROSS ALL DATASETS ===\n")
print(all_fairness.to_string(index=False))

all_fairness.to_csv(
    r'C:\Users\niyat\Desktop\pr\ML-Bias-Fairness\results\all_fairness_metrics.csv',
    index=False)

print("\n✅ ALL FAIRNESS METRICS COMPLETE")
print("Results saved to results/all_fairness_metrics.csv")

=== COMPLETE FAIRNESS METRICS ACROSS ALL DATASETS ===

      Dataset   Model Protected     SPD     DI     EOD     AOD
        Adult XGBoost    Gender -0.1963 0.3580 -0.0957 -0.0835
German Credit XGBoost    Gender -0.0748 0.8966 -0.0667 -0.1444
   Law School XGBoost    Gender -0.0212 0.9768 -0.0006 -0.0203

✅ ALL FAIRNESS METRICS COMPLETE
Results saved to results/all_fairness_metrics.csv
